# Bronze Layer — Raw Data Ingestion

Explore the `bronze` Delta tables:
- Raw CSV data ingested into Delta Lake
- Lineage columns: `_ingestion_timestamp`, `_source_system`, `_load_date`, `_source_file`
- Append-only, immutable log

In [1]:
%pip install pyspark==3.5.3

  Using cached pyspark-3.5.3-py2.py3-none-any.whl
  Attempting uninstall: pyspark
    Found existing installation: pyspark 3.5.0
    Can't uninstall 'pyspark'. No files were found to uninstall.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
spark = SparkSession.builder \
    .appName("explore_bronze") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://dataops-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "dataops-key") \
    .config("spark.hadoop.fs.s3a.secret.key", "dataops-secret") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

print(f"Spark {spark.version} ready")

Spark 3.5.0 ready


In [3]:
BASE = "s3a://delta-lake/bronze"

tables = ["customers", "products", "transactions"]
for t in tables:
    df = spark.read.format("delta").load(f"{BASE}/{t}")
    print(f"\n{'='*60}")
    print(f"{t.upper()}: {df.count()} rows \u00d7 {len(df.columns)} cols")
    print(f"Columns: {df.columns}")
    df.show(5, truncate=False)


CUSTOMERS: 100 rows × 14 cols
Columns: ['customer_id', 'first_name', 'last_name', 'email', 'phone', 'city', 'state', 'zip_code', 'created_date', 'is_active', '_ingestion_timestamp', '_source_system', '_load_date', '_source_file']
+-----------+----------+---------+--------------------------+------------+------------+-----+--------+------------+---------+--------------------------+--------------+----------+-----------------------+
|customer_id|first_name|last_name|email                     |phone       |city        |state|zip_code|created_date|is_active|_ingestion_timestamp      |_source_system|_load_date|_source_file           |
+-----------+----------+---------+--------------------------+------------+------------+-----+--------+------------+---------+--------------------------+--------------+----------+-----------------------+
|CUST_000001|William   |Johnson  |william.johnson0@gmail.com|+16413283286|Los Angeles |TX   |21395   |2025-08-01  |false    |2026-06-30 07:49:32.695937|ecommerc

In [4]:
# Check lineage columns
df = spark.read.format("delta").load(f"{BASE}/customers")
lin_cols = [c for c in df.columns if c.startswith("_ingestion") or c.startswith("_source") or c.startswith("_load")]
print("Lineage columns:", lin_cols)
df.select(*lin_cols).distinct().show(truncate=False)

Lineage columns: ['_ingestion_timestamp', '_source_system', '_load_date', '_source_file']
+--------------------------+--------------+----------+-----------------------+
|_ingestion_timestamp      |_source_system|_load_date|_source_file           |
+--------------------------+--------------+----------+-----------------------+
|2026-06-30 07:49:32.695937|ecommerce_api |2026-06-30|/data/raw/customers.csv|
+--------------------------+--------------+----------+-----------------------+



In [5]:
# Data quality: null primary keys
pk_map = {"customers": "customer_id", "products": "product_id", "transactions": "transaction_id"}
for t, pk in pk_map.items():
    df = spark.read.format("delta").load(f"{BASE}/{t}")
    nulls = df.filter(f"{pk} IS NULL").count()
    dups = df.count() - df.select(pk).distinct().count()
    print(f"{t}: {nulls} null {pk}, {dups} duplicates")

customers: 0 null customer_id, 0 duplicates
products: 0 null product_id, 0 duplicates
transactions: 0 null transaction_id, 0 duplicates


In [6]:
spark.stop()